# Source E Draft: Medi-Cal FFS provider availability

This is a separate, unexecuted draft. The original `source_e_analysis.ipynb` is not changed. The priority is to count enrolled FFS providers as accurately as possible at their service locations, while keeping provider records, specialties, locations, and billing status distinct.

**Important limitation:** the FFS provider directory does not provide a count of Medi-Cal patients served by each location. A provider's `Medicaid_Patients` field describes whether they accept new patients, not how many patients are enrolled or served. To calculate FFS providers per 1,000 patients, add a separate, date-aligned Medi-Cal member/enrollment denominator by geography. Until then, this notebook will produce provider counts only and will not invent a rate.

Run cells from top to bottom. Review the assumptions and output checks before using results.

## Source references

- Existing exploratory notebook: `source_e_analysis.ipynb`
- FFS provider CSV part 1: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/zipped_bc_too_big_for_github/medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv`
- FFS provider CSV part 2: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/zipped_bc_too_big_for_github/medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv`
- Dataset metadata: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/datapackage.json`
- FI provider type reference: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/fi-provider-type-code-reference.csv`
- FI specialty reference: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/fi-provider-specialty-code-reference.csv`
- IEHP NPI-deduplicated provider/match file: `data/provider-side/E-mediCal-fee-for-service-enrolled-providers/07_UNIQUE_IEHP_PROVIDERS_DEDUPED_BY_NPI.csv`

Metadata describes the FFS file as a point-in-time enrollment/geographic listing (as of October 26, 2026) and its coverage includes in- and out-of-state locations. Confirm the snapshot date and scope before comparing it with IEHP or member counts.

## Counting plan and cautions

- Preserve identifiers and postal codes as text; do not aggregate provider rows as though each row were a different provider.
- The file contains repeated providers across specialties and service locations. Report both **unique provider identities** and **provider-location records**. The location-level count is usually more useful for access, but is not a count of clinicians or patient capacity.
- Use NPI as the main identity where it is present and valid; use `Provider_Number` only as a documented fallback. Keep a count of fallback/invalid identifiers. Do not silently drop records.
- De-duplicate specialty repetitions only at a provider + owner + service-location + address key. Inspect key collisions and missing location fields before relying on counts.
- Keep FFS billing and non-billing providers separately visible. The full enrolled-provider count and the actively enrolled FFS billing-provider count answer different questions.
- For geographic summaries, retain both the source-reported `County` and coordinate-derived `CountyName`; the analysis county is selected with a clearly stated preference and should be checked for disagreement. ZIP is text.
- A listed provider is not necessarily accepting new patients, available to a particular IEHP member, or delivering a particular volume of care. Counts are directory-based supply indicators, not utilization or capacity.

In [ ]:
from pathlib import Path
import pandas as pd
import re

relative_data_dir = Path('data/provider-side/E-mediCal-fee-for-service-enrolled-providers')
repo_root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / relative_data_dir).is_dir()), None)
if repo_root is None:
    raise FileNotFoundError('Could not locate the Source E data directory. Start Jupyter in the repository or a subfolder.')

source_dir = repo_root / relative_data_dir
partition_dir = source_dir / 'zipped_bc_too_big_for_github'
ffs_files = [
    partition_dir / 'medi-cal-ffs-provider-listing_10_26_26_PART 1 first 200k line.csv',
    partition_dir / 'medi-cal-ffs-provider-listing_10_26_26_PART 2 next 180k line.csv',
]
missing_files = [str(p) for p in ffs_files if not p.is_file()]
if missing_files:
    raise FileNotFoundError('Missing FFS CSV part(s): ' + '; '.join(missing_files))

# Read all columns as strings so identifiers, codes, ZIPs, and phone numbers retain formatting.
read_options = dict(encoding='cp1252', dtype='string', keep_default_na=False, na_values=[''], low_memory=False)
providers_df = pd.concat([pd.read_csv(path, **read_options) for path in ffs_files], ignore_index=True)
print(f'Loaded {len(providers_df):,} source rows and {providers_df.shape[1]} columns')
print('Columns:', providers_df.columns.tolist())


## Initial integrity checks

Check the combined parts, key fields, placeholders, and repeated records before selecting an aggregation grain. This cell intentionally does not delete records.

In [ ]:
print('Shape:', providers_df.shape)
print('Duplicate full rows:', int(providers_df.duplicated().sum()))
key_columns = [c for c in ['Provider_Number', 'NPI', 'Owner_Number', 'Service_Location_Number', 'Legal_Name', 'Address', 'City', 'State', 'ZIP', 'County', 'CountyName', 'FI_Provider_Type', 'FI_Provider_Specialty', 'Billing_Provider'] if c in providers_df.columns]
display(providers_df[key_columns].head(10))
display(providers_df[key_columns].isna().sum().sort_values(ascending=False).to_frame('blank_fields'))


## Normalize identity and location fields

Normalization below is deliberately conservative: trim whitespace, preserve source values, and use NPI only when it is a 10-digit string. The provider identity key marks whether it came from NPI or the legacy/provider number. A fallback key is not equivalent to a verified NPI.

In [ ]:
def clean_text(series):
    return series.astype('string').str.strip().replace('', pd.NA)

for col in ['NPI', 'Provider_Number', 'Owner_Number', 'Service_Location_Number', 'Legal_Name', 'Address', 'Address2', 'City', 'State', 'ZIP', 'ZIP_4', 'County', 'CountyName', 'Billing_Provider']:
    if col in providers_df.columns:
        providers_df[col + '_clean'] = clean_text(providers_df[col])

npi_digits = providers_df['NPI_clean'].str.fullmatch(r'\d{10}', na=False)
providers_df['provider_id'] = pd.NA
providers_df.loc[npi_digits, 'provider_id'] = 'NPI:' + providers_df.loc[npi_digits, 'NPI_clean']
fallback = ~npi_digits & providers_df['Provider_Number_clean'].notna()
providers_df.loc[fallback, 'provider_id'] = 'PROVIDER_NUMBER:' + providers_df.loc[fallback, 'Provider_Number_clean']

def normalize_address(series):
    return (series.astype('string').str.upper().str.replace(r'\s+', ' ', regex=True).str.strip().replace('', pd.NA))

providers_df['address_key'] = normalize_address(providers_df.get('Address_clean', pd.Series(pd.NA, index=providers_df.index)))
providers_df['address2_key'] = normalize_address(providers_df.get('Address2_clean', pd.Series(pd.NA, index=providers_df.index)))
providers_df['city_key'] = normalize_address(providers_df.get('City_clean', pd.Series(pd.NA, index=providers_df.index)))
providers_df['state_key'] = normalize_address(providers_df.get('State_clean', pd.Series(pd.NA, index=providers_df.index)))
providers_df['zip_key'] = providers_df.get('ZIP_clean', pd.Series(pd.NA, index=providers_df.index)).astype('string').str.replace(r'\.0$', '', regex=True)

providers_df['analysis_county'] = providers_df.get('CountyName_clean', pd.Series(pd.NA, index=providers_df.index)).combine_first(providers_df.get('County_clean', pd.Series(pd.NA, index=providers_df.index)))
providers_df['analysis_county'] = providers_df['analysis_county'].astype('string').str.strip().str.replace(r'\s+County$', '', case=False, regex=True)

print('Valid 10-digit NPI rows:', int(npi_digits.sum()))
print('Rows using Provider_Number fallback:', int(fallback.sum()))
print('Rows with no usable provider identifier:', int(providers_df['provider_id'].isna().sum()))
print('Rows missing ZIP:', int(providers_df['zip_key'].isna().sum()))
print('Rows missing both county fields:', int(providers_df['analysis_county'].isna().sum()))


## Build provider-location records

This collapses repeated specialty rows for the same provider and service location while retaining distinct locations. The key includes owner/location numbers and normalized address fields, so source inconsistencies may still split one real location or merge records imperfectly. Review the diagnostics and, if necessary, refine the key using source documentation. Records without an identifier are retained in a separate review table rather than silently counted as unique providers.

In [ ]:
id_rows = providers_df.loc[providers_df['provider_id'].notna()].copy()
location_key_cols = ['provider_id']
for col in ['Owner_Number_clean', 'Service_Location_Number_clean', 'address_key', 'address2_key', 'city_key', 'state_key', 'zip_key']:
    if col in id_rows.columns:
        location_key_cols.append(col)

# Explicit sentinel prevents nulls in key components from being treated ambiguously.
id_rows['provider_location_key'] = id_rows[location_key_cols].fillna('<MISSING>').astype('string').agg(' | '.join, axis=1)

def any_billing(values):
    return values.astype('string').str.strip().eq('FFS Billing Provider').any()

agg_spec = {
    'provider_id': ('provider_id', 'first'),
    'NPI_clean': ('NPI_clean', 'first'),
    'Provider_Number_clean': ('Provider_Number_clean', 'first'),
    'analysis_county': ('analysis_county', 'first'),
    'zip_key': ('zip_key', 'first'),
    'City_clean': ('City_clean', 'first'),
    'State_clean': ('State_clean', 'first'),
    'Legal_Name_clean': ('Legal_Name_clean', 'first'),
    'has_billing_row': ('Billing_Provider_clean', any_billing),
    'source_rows': ('provider_id', 'size'),
    'specialties': ('FI_Provider_Specialty', lambda s: ' | '.join(sorted(set(s.dropna().astype(str))))) if 'FI_Provider_Specialty' in id_rows.columns else ('provider_id', 'size'),
}
provider_locations = id_rows.groupby('provider_location_key', dropna=False).agg(**agg_spec).reset_index()

print('Source rows with an identifier:', f'{len(id_rows):,}')
print('Provider-location records after specialty-row collapse:', f'{len(provider_locations):,}')
print('Unique provider identity keys:', f"{provider_locations['provider_id'].nunique():,}")
print('Provider-location keys with multiple source rows:', int((provider_locations['source_rows'] > 1).sum()))
print('Provider locations with billing designation:', int(provider_locations['has_billing_row'].sum()))
display(provider_locations.sort_values('source_rows', ascending=False).head(10))

# Rows lacking both NPI and Provider_Number are excluded from keyed counts, but retained here for review.
unkeyed_rows = providers_df.loc[providers_df['provider_id'].isna()].copy()
print('Unkeyed rows reserved for review:', f'{len(unkeyed_rows):,}')

## Geographic supply summaries

These are distinct provider-location counts and distinct provider identities represented in each geography. A provider working in multiple ZIPs or counties contributes to each applicable geography. Therefore county/ZIP sums may exceed the statewide unique-provider total. Compare the source county and coordinate-derived county fields before treating county assignment as definitive.

In [ ]:
geo = provider_locations.copy()
geo['analysis_county'] = geo['analysis_county'].astype('string').str.strip()
geo['zip_key'] = geo['zip_key'].astype('string').str.strip()

def summarize_geo(frame, group_cols):
    return (frame.dropna(subset=group_cols)
            .groupby(group_cols, dropna=False)
            .agg(ffs_provider_locations=('provider_location_key', 'nunique'),
                 unique_provider_identities=('provider_id', 'nunique'),
                 billing_provider_locations=('has_billing_row', 'sum'))
            .reset_index()
            .sort_values(['ffs_provider_locations'] + group_cols, ascending=[False] + [True] * len(group_cols)))

county_summary = summarize_geo(geo, ['analysis_county'])
county_zip_summary = summarize_geo(geo, ['analysis_county', 'zip_key'])
display(county_summary.head(20))
display(county_zip_summary.head(30))

print('County labels in provider locations:', geo['analysis_county'].nunique(dropna=True))
print('ZIP labels in provider locations:', geo['zip_key'].nunique(dropna=True))
print('Provider-location records lacking county:', int(geo['analysis_county'].isna().sum()))
print('Provider-location records lacking ZIP:', int(geo['zip_key'].isna().sum()))


## Optional comparison with IEHP NPI-deduplicated file

The IEHP CSV appears to summarize IEHP provider records and their FFS match classification. It is not a substitute for the full Source E provider-location file. Use it as a cross-check or focused IEHP subset, not as the total number of FFS providers. This code loads it only if available in the checked-out branch. The current draft branch may need that CSV copied/merged into it before this optional cell can run.

In [ ]:
iehp_path = source_dir / '07_UNIQUE_IEHP_PROVIDERS_DEDUPED_BY_NPI.csv'
if iehp_path.is_file():
    iehp_df = pd.read_csv(iehp_path, encoding='utf-8-sig', dtype='string', keep_default_na=False, na_values=[''])
    print('IEHP rows:', f'{len(iehp_df):,}')
    print('IEHP columns:', iehp_df.columns.tolist())
    if 'ffs_match_classification' in iehp_df.columns:
        display(iehp_df['ffs_match_classification'].value_counts(dropna=False).to_frame('rows'))
    else:
        print('Expected match-classification field not found; inspect headers before using this file.')
else:
    print(f'Optional IEHP file not present in this checkout: {iehp_path}')


## Providers per 1,000 Medi-Cal patients: denominator required

To calculate a rate, obtain a trustworthy number of enrolled Medi-Cal members/patients for the **same snapshot period and geography** (for example, Riverside and San Bernardino county or ZIP). The denominator must align with the provider numerator's population and geographic definitions. An IEHP-specific member count should not be used as the denominator for all Medi-Cal FFS providers unless the intended population is explicitly IEHP members and the numerator is restricted accordingly.

Formula: `FFS provider locations / enrolled patients * 1,000`. This is provider locations per 1,000 patients, not patients per provider. The optional code below runs only after a denominator CSV is supplied with `analysis_county` and `enrolled_patients` columns.

In [ ]:
# Set this to a reviewed, date-aligned member-count CSV when one is available.
patient_counts_path = None  # Example: source_dir / 'medi_cal_enrollment_by_county.csv'

if patient_counts_path is None:
    print('No patient denominator supplied: provider-per-1,000-patient rates are intentionally not calculated.')
else:
    patient_counts = pd.read_csv(patient_counts_path, dtype='string', keep_default_na=False, na_values=[''])
    required = {'analysis_county', 'enrolled_patients'}
    if not required.issubset(patient_counts.columns):
        raise ValueError(f'Denominator file must contain columns {sorted(required)}')
    patient_counts['analysis_county'] = patient_counts['analysis_county'].astype('string').str.strip().str.replace(r'\s+County$', '', case=False, regex=True)
    patient_counts['enrolled_patients'] = pd.to_numeric(patient_counts['enrolled_patients'], errors='coerce')
    if patient_counts['analysis_county'].duplicated().any():
        raise ValueError('Denominator must have one row per analysis_county, or be aggregated using documented rules first.')
    rates = county_summary.merge(patient_counts, on='analysis_county', how='left', validate='one_to_one')
    rates['ffs_provider_locations_per_1000_patients'] = (rates['ffs_provider_locations'] / rates['enrolled_patients']) * 1000
    display(rates.sort_values('ffs_provider_locations_per_1000_patients', ascending=False))


## Interpretation checklist

Before reporting any result: (1) verify both source partitions and row totals, (2) examine invalid/missing NPIs and fallback IDs, (3) manually inspect high-collision provider-location keys, (4) compare county and coordinate-derived county, (5) report all enrolled vs. billing-only counts separately, (6) document the geographic counting rule, (7) verify denominator source/date/population, and (8) describe the result as directory-listed supply rather than actual availability or care delivered. No final interpretation is recorded in this draft yet.